# Reporting And API Contract Walkthrough

This notebook focuses on the read/reporting side of the repo. It uses the `shiny_bird_foil_only` fixture because that bundle creates a nice reporting example: a normal-finish inventory row with only foil pricing available.


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root. Start Jupyter from somewhere inside the repo.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
FIXTURES_DIR = REPO_ROOT / "tests" / "fixtures"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def reset_dir(path: Path) -> Path:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def copy_fixture_bundle(destination: Path, fixture_name: str, *filenames: str) -> dict[str, Path]:
    destination.mkdir(parents=True, exist_ok=True)
    copied: dict[str, Path] = {}
    for filename in filenames:
        source = FIXTURES_DIR / fixture_name / filename
        target = destination / filename
        shutil.copyfile(source, target)
        copied[filename] = target
    return copied

from mtg_source_stack.api_contract import api_error_payload, api_error_status
from mtg_source_stack.db.schema import initialize_database
from mtg_source_stack.importer.mtgjson import import_mtgjson_identifiers, import_mtgjson_prices
from mtg_source_stack.importer.scryfall import import_scryfall_cards
from mtg_source_stack.inventory.response_models import serialize_response
from mtg_source_stack.inventory.service import (
    add_card,
    create_inventory,
    export_inventory_csv,
    inventory_health,
    inventory_report,
    list_owned_filtered,
    list_price_gaps,
    reconcile_prices,
    valuation_filtered,
)

WORKDIR = reset_dir(REPO_ROOT / "var" / "walkthrough" / "04_reporting_and_api_contract")
DB_PATH = WORKDIR / "collection.db"
EXPORT_PATH = WORKDIR / "personal_export.csv"
fixture_bundle = copy_fixture_bundle(
    WORKDIR,
    "shiny_bird_foil_only",
    "scryfall.json",
    "identifiers.json",
    "prices.json",
)
initialize_database(DB_PATH)
import_scryfall_cards(DB_PATH, fixture_bundle["scryfall.json"])
import_mtgjson_identifiers(DB_PATH, fixture_bundle["identifiers.json"])
import_mtgjson_prices(DB_PATH, fixture_bundle["prices.json"])
create_inventory(DB_PATH, slug="personal", display_name="Personal Collection", description="Reporting demo")
add_card(
    DB_PATH,
    inventory_slug="personal",
    inventory_display_name=None,
    scryfall_id="foil-only-1",
    tcgplayer_product_id=None,
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=1,
    condition_code="NM",
    finish="normal",
    language_code="en",
    location="",
    acquisition_price=None,
    acquisition_currency=None,
    notes=None,
    tags=None,
)

def pretty(value):
    print(json.dumps(serialize_response(value), indent=2))


## Owned Rows And Valuation

Even before a row is perfectly priced, the reporting surface can still list it and compute valuation summaries from the latest matching price data.


In [ ]:
owned = list_owned_filtered(
    DB_PATH,
    inventory_slug="personal",
    provider="tcgplayer",
    limit=None,
    query=None,
    set_code=None,
    rarity=None,
    finish=None,
    condition_code=None,
    language_code=None,
    location=None,
    tags=None,
)
valuation = valuation_filtered(
    DB_PATH,
    inventory_slug="personal",
    provider="tcgplayer",
    query=None,
    set_code=None,
    rarity=None,
    finish=None,
    condition_code=None,
    language_code=None,
    location=None,
    tags=None,
)
print("Owned rows:")
pretty(owned)
print("\nValuation rows:")
pretty(valuation)


## Price Gaps, Reconcile Suggestions, And Health Checks

This fixture intentionally produces a price-gap scenario: the inventory row is `normal`, but pricing only exists for `foil`.


In [ ]:
price_gaps = list_price_gaps(DB_PATH, inventory_slug="personal", provider="tcgplayer", limit=None)
reconcile_preview = reconcile_prices(
    DB_PATH,
    inventory_slug="personal",
    provider="tcgplayer",
    apply_changes=False,
)
health = inventory_health(
    DB_PATH,
    inventory_slug="personal",
    provider="tcgplayer",
    stale_days=30,
    preview_limit=5,
)
print("Price gaps:")
pretty(price_gaps)
print("\nReconcile preview:")
pretty(reconcile_preview)
print("\nInventory health:")
pretty(health)


## Inventory Report And CSV Export

`inventory_report()` combines the row list, valuation, and health summaries. `export_inventory_csv()` writes the filtered owned-row view to disk.


In [ ]:
report = inventory_report(
    DB_PATH,
    inventory_slug="personal",
    provider="tcgplayer",
    query=None,
    set_code=None,
    rarity=None,
    finish=None,
    condition_code=None,
    language_code=None,
    location=None,
    tags=None,
    limit=5,
    stale_days=30,
)
export_result = export_inventory_csv(
    DB_PATH,
    inventory_slug="personal",
    provider="tcgplayer",
    output_path=EXPORT_PATH,
    query=None,
    set_code=None,
    rarity=None,
    finish=None,
    condition_code=None,
    language_code=None,
    location=None,
    tags=None,
    limit=None,
)
print("Report summary:")
pretty(report.summary)
print("\nExport result:")
pretty(export_result)
print(f"\nCSV exists: {EXPORT_PATH.exists()}")


## API-Facing Serialization And Error Mapping

The current demo web API shell serializes service results and maps domain exceptions into a small HTTP-facing error envelope.

The HTTP contract now includes explicit OpenAPI response models, echoes `X-Request-Id`, defaults audit writes to the `local-demo` actor unless trusted-header mode is enabled, and keeps `/health` focused on status and mode rather than filesystem path details.


In [ ]:
print("Serialized report payload excerpt:")
pretty({
    "summary": report.summary,
    "valuation_rows": report.valuation_rows,
})

try:
    reconcile_prices(
        DB_PATH,
        inventory_slug="personal",
        provider="tcgplayer",
        apply_changes=True,
    )
except Exception as exc:
    print("\nMapped error status:", api_error_status(exc))
    print(json.dumps(api_error_payload(exc), indent=2))


## Wrap-Up

At this point you have seen the full repository arc: repo structure, database initialization, source imports, inventory-domain mutations, and the reporting/API-facing read surface built on top of the same local SQLite runtime.
